# Opioid Use Disorder (OUD) Risk Prediction
## Exploratory notebook sandbox

This notebook is an exploratory sandbox for Machine Learning Operations (MLOps) development

Use it to
- Inspect raw and cleaned data
- Perform focused Exploratory Data Analysis (EDA) checks to validate assumptions
- Run quick training and inference checks while iterating on features
- Experiment with new features and models to improve metrics performance

Do not use it to
- Replace the reproducible pipeline in `src/main.py`
- Store production logic in notebook cells
- Write production artifacts to disk from the notebook

Canonical production entry point
- Run from terminal: `python -m src.main`

Why this separation matters
- The orchestrator (`src/main.py`) is the factory: deterministic inputs, deterministic outputs, clear provenance
- The notebook is the lab bench: interactive inspection, rapid iteration, visible intermediate states and exploration

### A) Environment setup and imports

Learning intent
- Make imports reliable regardless of how you opened Jupyter
- Ensure relative paths behave the same way for everyone
- Keep notebook code thin by calling functions from `src/` modules

In [1]:
%load_ext autoreload
%autoreload 2

Failed to read module file 'C:\Users\carne\miniconda3\envs\mlops-student-env\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\carne\miniconda3\envs\mlops-student-env\Lib\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "C:\Users\carne\miniconda3\envs\mlops-student-env\Lib\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "C:\Users\carne\miniconda3\envs\mlops-student-env\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1398, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1371, in _find_and_load
  File "<frozen importlib._bootstrap>", line 1335, in _find_and_load_unloc

In [2]:
from __future__ import annotations

import os # For changing the working directory to the repo root
import sys # For manipulating sys.path to ensure imports work regardless of where the notebook is started
from pathlib import Path # For working with file paths in a platform-independent way

import pandas as pd
from IPython.display import display # For displaying dataframes nicely in Jupyter

# Repo root alignment
# Problem this solves
# - Notebooks can start with different working directories depending on IDE and settings
# - Relative paths then break unpredictably
#
# Approach
# - Search upward from the current folder until we find a folder that contains `src/`
# - Set that as the working directory so relative paths match the orchestrator behaviour
def find_repo_root(start: Path, marker_dir: str = "src", max_hops: int = 12) -> Path:
    current = start.resolve()
    for _ in range(max_hops):
        if (current / marker_dir).exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    raise RuntimeError(
        f"Could not find repo root containing '{marker_dir}/' starting from: {start}"
    )

PROJECT_ROOT = find_repo_root(Path.cwd())
os.chdir(PROJECT_ROOT)

# Ensure `import src...` works even if Jupyter started elsewhere
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)
print("cwd:", Path.cwd())

# Imports from production modules
# Note: we intentionally do NOT import from src/main.py
from src.load_data import load_raw_data
from src.clean_data import clean_dataframe
from src.validate import validate_dataframe
from src.features import get_feature_preprocessor
from src.train import train_model
from src.evaluate import evaluate_model
from src.infer import run_inference

PROJECT_ROOT: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo
cwd: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo


### B) Sandbox configuration

Guideline
- Keep configuration in one place
- Use the same column names and defaults as the orchestrator where possible
- Prefer explicit lists over clever inference so errors are actionable

In [ ]:
SETTINGS = {
    "is_example_config": False,
    "problem_type": "regression",
    "random_state": 42,
    "test_size": 0.25,
    "target_column": "Rent",
    "paths": {
        "raw_data": "data/raw/dataset.csv",
        "processed_data": "data/processed/clean.csv",
        "model": "models/model.joblib",
        "predictions": "reports/predictions.csv",
    },
    "features": {
        # Pre-configured to match the dummy CSV created by src/load_data.py
        "quantile_bin": [],  # keep empty by default; dummy numeric feature passes through
        "categorical_onehot": ["District"],
        "numeric_passthrough": ["Sq.Mt","Floor","Bedrooms","Outer","Duplex","Cottage","Elevtor","Penthouse","Semidettached"],
        "n_bins": 3,
    },
}

def three_way_split(
    X: pd.DataFrame,
    y: pd.Series,
    *,
    test_size: float,
    val_size: float,
    random_state: int,
    stratify: bool,
):
    """
    Sandbox copy aligned with the orchestrator intent
    - Students benefit from seeing the leakage gate explicitly
    - We avoid importing helpers from src/main.py to prevent side effects

    Behaviour
    - Try stratified splits for classification
    - If stratification fails, fall back to random splits with a clear message
    """
    from sklearn.model_selection import train_test_split

    if test_size <= 0 or val_size <= 0 or (test_size + val_size) >= 1.0:
        raise ValueError("Split sizes must satisfy 0 < test_size, 0 < val_size, and test_size + val_size < 1")

    stratify_y = y if stratify else None

    try:
        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=random_state,
            stratify=stratify_y,
        )

        relative_val_size = val_size / (1.0 - test_size)
        stratify_temp = y_temp if stratify else None

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp,
            test_size=relative_val_size,
            random_state=random_state,
            stratify=stratify_temp,
        )

        return X_train, X_val, X_test, y_train, y_val, y_test

    except ValueError as e:
        print(f"[notebook] Stratified split failed: {e}")
        print("[notebook] Falling back to random split")

        X_temp, X_test, y_temp, y_test = train_test_split(
            X, y,
            test_size=test_size,
            random_state=random_state,
        )

        relative_val_size = val_size / (1.0 - test_size)

        X_train, X_val, y_train, y_val = train_test_split(
            X_temp, y_temp,
            test_size=relative_val_size,
            random_state=random_state,
        )

        return X_train, X_val, X_test, y_train, y_val, y_test

print("RAW_DATA_PATH:", RAW_DATA_PATH)
print("TARGET:", SETTINGS["target_column"])
print("PROBLEM_TYPE:", SETTINGS["problem_type"])

RAW_DATA_PATH: /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/data/raw/opiod_raw_data.csv
TARGET: OD
PROBLEM_TYPE: classification


## 1) Load raw data (`src.load_data`)

### Role  
Load raw CSV data in a controlled and reproducible manner.

### Input  
- `raw_data_path: Path`

### Output  
- `df_raw: pandas.DataFrame`

### Behavior  

- Logs the source path (temporary `print`, to be replaced with `logging`).  
- If the file is missing:
  - In example mode (`IS_EXAMPLE_CONFIG=true`), creates a deterministic dummy dataset.
  - Otherwise, raises `FileNotFoundError`.  
- Raises `ValueError` if the loaded DataFrame is empty.  

### Contract  

Guarantees deterministic loading and explicit failure on invalid or missing data, supporting reproducible ML pipelines.

In [ ]:
import os
from pathlib import Path

import pandas as pd

from src.utils import load_csv, save_csv


def load_raw_data(raw_data_path: Path) -> pd.DataFrame:
    """
    Inputs:
    - raw_data_path: Path to the raw CSV file.
    Outputs:
    - df_raw: Raw DataFrame loaded from disk.
    Why this contract matters for reliable ML delivery:
    - “Same inputs, same outputs” is the foundation of reproducible ML
      pipelines.
    """
    # TODO: replace with logging later
    print(f"[load_data.load_raw_data] Loading raw data from: {raw_data_path}")

    is_example_config = (
        os.getenv("IS_EXAMPLE_CONFIG", "true").lower() == "true"
    )

    if not raw_data_path.exists():
        if is_example_config:
            raw_data_path.parent.mkdir(parents=True, exist_ok=True)

            dummy = pd.DataFrame(
                {
                    "num_feature": [0.0, 1.0, 2.0, 3.0, 4.0, 5.0],
                    "cat_feature": ["A", "B", "A", "C", "B", "C"],
                    "target": [0.0, 1.2, 1.9, 3.1, 3.8, 5.2],
                }
            )

            # TODO: replace with logging later
            print(
                "\n"
                "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n"
                "LOUD WARNING (SCAFFOLDING ONLY):\n"
                f"- Raw data file was not found at: {raw_data_path}\n"
                "A tiny deterministic DUMMY dataset was created with:\n"
                'Columns:  ["num_feature", "cat_feature", "target"]\n'
                "ONLY to ensure the pipeline runs end-to-end immediately.\n"
                "MUST replace this dataset + update SETTINGS in src/main.py.\n"
                "!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!!\n"
                )

            save_csv(dummy, raw_data_path)
        else:
            raise FileNotFoundError(
                "[load_data.load_raw_data] Raw data file not found.\n"
                f"Expected at: {raw_data_path}\n"
                "Fix:\n"
                "1) Put your dataset at that path, OR\n"
                "2) Update SETTINGS['raw_data_path'] (and later config.yml) to"
                "the correct file.\n"
            )

    df_raw = load_csv(raw_data_path)

    if df_raw is None or df_raw.empty:
        raise ValueError(
            "[load_data.load_raw_data] Loaded dataframe is empty.\n"
            f"File path: {raw_data_path}\n"
            "Fix:\n"
            "Check the file contents (maybe header-only or wrong delimiter)\n"
            "Confirm your export/query actually produced rows.\n"
        )

    # TODO: replace with logging later
    print(
        "[load_data.load_raw_data] Loaded shape=%s, columns=%s"
        % (df_raw.shape, list(df_raw.columns))
    )

    # --------------------------------------------------------
    # START STUDENT CODE
    # --------------------------------------------------------
    # TODO_STUDENT: Paste your notebook logic here to replace or extend the
    # baseline
    # Why: Explain why this step varies per dataset or business context
    # Examples:
    # 1. ...
    # 2. ...
    #
    # Optional forcing function (leave commented)
    # raise NotImplementedError("Student: You must implement this logic to
    # proceed!")
    #
    # Placeholder (Remove this after implementing your code):
    print("Warning: Student has not implemented this section yet")
    # --------------------------------------------------------
    # END STUDENT CODE
    # --------------------------------------------------------

    return df_raw

[load_data.load_raw_data] Loading raw data from /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/data/raw/opiod_raw_data.csv
[utils.load_csv] Loading CSV from /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/data/raw/opiod_raw_data.csv
[load_data.load_raw_data] Loaded dataframe shape: (1000, 22)
df_raw.shape: (1000, 22)


,ID,OD,Low_inc,SURG,rx ds,A,B,C,D,E,...,I,J,K,L,M,N,R,S,T,V
0,1,1,1,0,330,0,0,0,1,1,...,1,1,0,1,1,1,1,0,0,0
1,2,1,1,0,457,0,0,1,0,1,...,1,1,0,0,1,1,1,1,0,0
2,3,1,1,1,722,0,1,0,1,1,...,1,1,1,0,1,0,1,0,1,0
3,4,0,1,0,262,0,1,0,0,1,...,1,0,1,1,1,1,1,1,1,0
4,5,1,1,0,780,0,0,0,0,1,...,1,0,0,0,1,0,1,0,0,0


## 3) Exploratory Data Analysis (EDA)

### Role  
Perform structured inspection of the cleaned dataset to understand data distribution, feature behavior, and potential modeling risks.

### Input  
- Cleaned `pandas.DataFrame`

### Output  
- Summary statistics and diagnostic plots (non-persistent, analysis-only)

### Objectives  

- Validate schema and column types  
- Inspect dataset shape and basic statistics  
- Analyze target distribution  
- Identify missing values or anomalies  
- Detect potential outliers or skewed features  

### Contract  

EDA is analytical and non-destructive.  
It does not modify data used in the pipeline; it informs modeling decisions while preserving reproducibility.

In [5]:
print("Missing values (top 10 columns):")
display(df_raw.isna().sum().sort_values(ascending=False).head(10))

target_col = SETTINGS["target_column"]
if target_col in df_raw.columns:
    vc = df_raw[target_col].value_counts(dropna=False)
    print("\nTarget distribution:")
    display(vc)
    if vc.sum() > 0:
        print("Positive class prevalence:", float(vc.get(1, 0) / vc.sum()))
else:
    print(f"\nTarget column '{target_col}' not found in raw data")

# Teaching variable: raw dataset may contain 'rx ds' or 'rx_ds'
rx_col = "rx_ds" if "rx_ds" in df_raw.columns else ("rx ds" if "rx ds" in df_raw.columns else None)
if rx_col is not None:
    print(f"\nSummary for '{rx_col}':")
    display(df_raw[rx_col].describe())

print("\nBinary indicator check (sample of 3 columns):")
for col in BINARY_SUM_COLS[:3]:
    if col in df_raw.columns:
        print(f"'{col}' unique values:", pd.Series(df_raw[col].unique()).head(10).tolist())

Missing values (top 10 columns):


ID         0
OD         0
Low_inc    0
SURG       0
rx ds      0
A          0
B          0
C          0
D          0
E          0
dtype: int64


Target distribution:


OD
0    819
1    181
Name: count, dtype: int64

Positive class prevalence: 0.181

Summary for 'rx ds':


count    1000.000000
mean      329.961000
std       333.335543
min         1.000000
25%        60.000000
50%       182.000000
75%       589.250000
max      1699.000000
Name: rx ds, dtype: float64


Binary indicator check (sample of 3 columns):
'A' unique values: [0, 1]
'B' unique values: [0, 1]
'C' unique values: [0, 1]


## 2) Data Cleaning (`src.clean_data`)

### Role  
Apply deterministic preprocessing and basic feature normalization to produce a cleaned dataset for downstream modeling.

### Input  
- `df_raw: pandas.DataFrame`  
- `target_column: str`

### Output  
- `df_clean: pandas.DataFrame`

### Behavior  

- Validates that the target column exists.  
- Removes duplicate rows.  
- Drops rows with missing target values.  
- Resets index after filtering.  
- Forces target column to numeric type (strict conversion).  
- Detects numeric and categorical columns.  
- Normalizes categorical features (strip + lowercase).  

### Contract  

Ensures controlled, reviewable, and deterministic cleaning while preserving pipeline stability.  
All transformations are explicit to reduce hidden state and modeling risk.

In [ ]:
import pandas as pd

# target_column is the dependent variable that we want to predict. It is used in cleaning to ensure we don't drop rows with missing target values, which would affect model training.
def clean_dataframe(df_raw: pd.DataFrame, target_column: str) -> pd.DataFrame:
    """
    Inputs:
    - df_raw: Raw pandas DataFrame
    - target_column: Name of the target column
    Outputs:
    - df_clean: Cleaned pandas DataFrame
    Why this contract matters for reliable ML delivery:
    - Cleaning changes data semantics; isolating it makes changes reviewable, testable, and less risky.
    """
    print("[clean_data.clean_dataframe] Cleaning raw dataframe (baseline = identity transform)")
    if target_column not in df_raw.columns:
        raise ValueError(f"Target column '{target_column}' not found in dataframe.")
    df_clean = df_raw.copy()
    df_clean.drop_duplicates(inplace=True)
    #drop rows with missing target values
    before_rows = len(df_clean)
    df_clean = df_clean[df_clean[target_column].notna()].copy()
    after_rows = len(df_clean)
    print(f"[clean_data.clean_dataframe] Dropped {before_rows - after_rows} rows with missing target")
    df_clean.reset_index(inplace=True)

    #force target column to be numeric (if not already)
    df_clean[target_column] = pd.to_numeric(df_clean[target_column], errors='raise')
    print("[clean_data.clean_dataframe] Target converted to numeric")

    #detect numeric columns
    numeric_columns = df_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()
    numeric_columns = [col for col in numeric_columns if col != target_column]

    #detect categorical columns
    categorical_columns = df_clean.select_dtypes(include=["object","string", "category"]).columns.tolist()

    #normalize categorical columns
    for col in categorical_columns:
        df_clean[col] = (df_clean[col].astype(str).str.strip().str.lower())
        print(f"[clean_data.clean_dataframe] Normalized categorical column {col}")

    print("Warning: Student has not implemented this section yet")


    return df_clean

    # --------------------------------------------------------
    # END STUDENT CODE
    # --------------------------------------------------------
    # Minimal baseline sanity: ensure target exists if referenced


[clean_data.clean_dataframe] Cleaning dataframe
[clean_data.clean_dataframe] Dropped 102 rows due to NA or duplicates
[clean_data.clean_dataframe] Rows after cleaning: 898
df_clean.shape: (898, 21)


,OD,Low_inc,SURG,rx_ds,A,B,C,D,E,F,...,I,J,K,L,M,N,R,S,T,V
0,1,1,0,330,0,0,0,1,1,1,...,1,1,0,1,1,1,1,0,0,0
1,1,1,0,457,0,0,1,0,1,0,...,1,1,0,0,1,1,1,1,0,0
2,1,1,1,722,0,1,0,1,1,0,...,1,1,1,0,1,0,1,0,1,0
3,0,1,0,262,0,1,0,0,1,1,...,1,0,1,1,1,1,1,1,1,0
4,1,1,0,780,0,0,0,0,1,1,...,1,0,0,0,1,0,1,0,0,0


### 4) Didactic check: what changed after cleaning

Goal
- Make transformations visible
- Help you debug unexpected column name changes or dropped columns

In [7]:
raw_cols = list(df_raw.columns)
clean_cols = list(df_clean.columns)

removed_cols = sorted(set(raw_cols) - set(clean_cols))
added_cols = sorted(set(clean_cols) - set(raw_cols))

print("Columns removed:")
display(removed_cols)

print("Columns added:")
display(added_cols)

print('Cleaned has "rx_ds":', "rx_ds" in df_clean.columns)

Columns removed:


['ID', 'rx ds']

Columns added:


['rx_ds']

Cleaned has "rx_ds": True


### 5) Validate data (security gate)

Educational note
- Validation is a fail-fast gate
- It prevents wasting time on training with broken assumptions

Guideline
- If validation fails, fix the upstream module or the data contract
- Do not patch around failures inside the notebook

In [8]:
required_columns = (
    [SETTINGS["target_column"]]
    + SETTINGS["features"]["quantile_bin"]
    + SETTINGS["features"]["categorical_onehot"]
    + SETTINGS["features"]["numeric_passthrough"]
    + SETTINGS["features"]["binary_sum_cols"]
)
required_columns = list(dict.fromkeys(required_columns))

validate_dataframe(
    df=df_clean,
    required_columns=required_columns,
    check_missing_values=SETTINGS["validation"]["check_missing_values"],
    target_column=SETTINGS["target_column"],
    target_allowed_values=[0, 1] if SETTINGS["problem_type"] == "classification" else None,
    numeric_non_negative_cols=SETTINGS["validation"]["numeric_non_negative_cols"],
)

print("[notebook] Validation passed")

[validate.validate_dataframe] Validating dataframe
[notebook] Validation passed


### 6) Build the feature recipe (`src.features`)

Educational note
- This step builds a preprocessing blueprint
- It must not fit on the full dataset in the notebook
- The recipe learns only when fitted on the training split inside the training pipeline

In [ ]:
from typing import List, Optional

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import KBinsDiscretizer, OneHotEncoder


def get_feature_preprocessor(
    quantile_bin_cols: Optional[List[str]] = None,
    categorical_onehot_cols: Optional[List[str]] = None,
    numeric_passthrough_cols: Optional[List[str]] = None,
    n_bins: int = 3,
):
    """
    Inputs:
    - quantile_bin_cols: Numeric columns to discretize into quantile bins
    - categorical_onehot_cols: Categorical columns to one-hot encode
    - numeric_passthrough_cols: Numeric columns to pass through unchanged
    - n_bins: Number of quantile bins
    Outputs:
    - preprocessor: Unfitted sklearn ColumnTransformer
    Why this contract matters for reliable ML delivery:
    - Keeping this as an unfitted recipe prevents leakage and ensures identical transforms at train and serve time.
    """
    print("[features.get_feature_preprocessor] Building ColumnTransformer recipe (unfitted)")  # TODO: replace with logging later

    quantile_bin_cols = quantile_bin_cols or []
    categorical_onehot_cols = categorical_onehot_cols or []
    numeric_passthrough_cols = numeric_passthrough_cols or []

    # Robust OneHotEncoder creation across sklearn versions
    try:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        ohe = OneHotEncoder(handle_unknown="ignore", sparse=False)

    quantile_pipe = Pipeline(
        steps=[
            ("kbins", KBinsDiscretizer(n_bins=n_bins, encode="onehot-dense", strategy="quantile")),
        ]
    )

    cat_pipe = Pipeline(
        steps=[
            ("onehot", ohe),
        ]
    )

    transformers = []
    if quantile_bin_cols:
        transformers.append(("quantile_bin", quantile_pipe, quantile_bin_cols))
    if categorical_onehot_cols:
        transformers.append(("categorical_onehot", cat_pipe, categorical_onehot_cols))
    if numeric_passthrough_cols:
        transformers.append(("numeric_passthrough", "passthrough", numeric_passthrough_cols))

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
    )

    # --------------------------------------------------------
    # START STUDENT CODE
    # --------------------------------------------------------
    # TODO_STUDENT: Modify feature recipe (scaling, imputation, text, interactions) as needed
    # Why: Feature engineering depends on data modality and business goals
    # Examples:
    # 1. Add SimpleImputer for missing numeric values
    # 2. Add StandardScaler for linear models
    #
    # Optional forcing function (leave commented)
    # raise NotImplementedError("Student: You must implement this logic to proceed!")
    #

    # --------------------------------------------------------
    # END STUDENT CODE
    # --------------------------------------------------------

    return preprocessor

[features.get_feature_preprocessor] Building feature recipe from configuration
[notebook] Feature recipe built, not fitted yet


,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('quantile_bins', ...), ('binary_sum', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``featu

### 7) Leakage gate: three-way split (train, validation, test)

Educational note
- Train split is the only split allowed to learn preprocessing parameters and model weights
- Validation split is for iteration and model selection
- Test split is a final audit vault

In [ ]:
X = df_clean.drop(columns=[SETTINGS["target_column"]])
y = df_clean[SETTINGS["target_column"]]

if SETTINGS["problem_type"] == "classification" and y.nunique() < 2:
    raise ValueError(
        f"Target '{SETTINGS['target_column']}' has fewer than 2 classes after cleaning\n"
        "Classification training and stratified splitting cannot proceed"
    )

X_train, X_val, X_test, y_train, y_val, y_test = three_way_split(
    X,
    y,
    test_size=SETTINGS["split"]["test_size"],
    val_size=SETTINGS["split"]["val_size"],
    random_state=SETTINGS["split"]["random_state"],
    stratify=(SETTINGS["problem_type"] == "classification"),
)

print("Train:", X_train.shape, "Validation:", X_val.shape, "Test:", X_test.shape)

if len(X_test) == 0:
    raise ValueError("Test split is empty. Check split ratios and dataset size.")

Train: (718, 20) Validation: (135, 20) Test: (45, 20)


### 8) Train (`src.train`)

Educational note
- This is the only step where `.fit()` happens
- The preprocessor and estimator are fitted only on training data

In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline


def train_model(X_train: pd.DataFrame, y_train: pd.Series, preprocessor, problem_type: str = "regression"):
    """
    Inputs:
    - X_train: Training features (DataFrame)
    - y_train: Training labels (Series)
    - preprocessor: ColumnTransformer (not fitted)
    - problem_type: "regression" or "classification"
    Outputs:
    - model: Fitted scikit-learn Pipeline
    Why this contract matters for reliable ML delivery:
    - Training is repeatable and leakage-resistant because preprocessing is fit only within the Pipeline on training data.
    """
    print(f"[train.train_model] Training model for problem_type={problem_type}")  # TODO: replace with logging later

    # --------------------------------------------------------
    # START STUDENT CODE
    # --------------------------------------------------------
    if X_train is None or len(X_train) == 0:
        raise ValueError("Training failed: X_train is empty.")

    if y_train is None or len(y_train) == 0:
        raise ValueError("Training failed: y_train is empty.")

    if len(y_train) != len(X_train):
        raise ValueError(
            f"Training failed: X_trian and y_train are different sized"
            f" X_train: {len(X_train)}, y_train: {len(y_train)}."
        )

    if problem_type == "regression":
        estimator = LinearRegression()
    else:
        raise ValueError(
            f"Training failed: problem_type: {problem_type} not supported"
         )

    model = Pipeline(
        steps=[
            ("preprocess", preprocessor),
            ("model", estimator),
        ]
    )

    model.fit(X_train, y_train)

    return model
    # --------------------------------------------------------
    # END STUDENT CODE
    # --------------------------------------------------------


[train.train_model] Training model pipeline for problem_type=classification
[notebook] Training complete


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocess', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('quantile_bins', ...), ('binary_sum', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different 

### 9) Evaluate (`src.evaluate`)

**Educational Note: The Audit Vault**
* **Validation:** Used to iteratively guide our model and feature engineering decisions.
* **Test:** Our blind audit vault. We use a strict boolean flag (`evaluate_on_test` in our `SETTINGS`) to ensure we only peek at this vault when we are absolutely ready to finalize the model. Notice that our production orchestrator (`src/main.py`) doesn't even contain code to score the test set!

**Metric Note**
* `evaluate_model` automatically selects the appropriate metrics based on the `problem_type` and returns them as a dictionary.
* **Classification:** Returns **PR AUC** (Average Precision - our primary metric for imbalanced data) and **ROC AUC**.
* **Regression:** Returns **RMSE** (Root Mean Squared Error).

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, Optional, Union

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


@dataclass(frozen=True)
class EvalConfig:
    reports_dir: Union[str, Path] = "reports"
    save_prefix: str = "eval"
    save_predictions_csv: bool = True
    # If you have an ID column in X_test (DataFrame), store it in the predictions CSV
    id_column: Optional[str] = None


def _ensure_dir(path: Union[str, Path]) -> Path:
    p = Path(path)
    p.mkdir(parents=True, exist_ok=True)
    return p


def _finite_mask(*arrays: np.ndarray) -> np.ndarray:
    mask = np.ones_like(arrays[0], dtype=bool)
    for a in arrays:
        mask &= np.isfinite(a)
    return mask


def evaluate(
    model: Any,
    X_test: Union[pd.DataFrame, np.ndarray],
    y_test: Union[pd.Series, np.ndarray],
    config: Optional[EvalConfig] = None,
) -> Dict[str, Any]:
    """
    Evaluate a trained regression model on test data.

    Saves to reports_dir:
      - *_pred_vs_actual.png
      - *_residuals_vs_pred.png
      - *_residual_hist.png
      - *_abs_error_hist.png
      - *_metrics.json
      - *_predictions.csv (optional)

    Returns:
      metrics dict
    """
    cfg = config or EvalConfig()
    reports_dir = _ensure_dir(cfg.reports_dir)

    y_true = np.asarray(y_test).reshape(-1)
    y_pred = np.asarray(model.predict(X_test)).reshape(-1)

    if y_true.shape[0] != y_pred.shape[0]:
        raise ValueError(f"y_test length {len(y_true)} != y_pred length {len(y_pred)}")

    # Remove non-finite values (protects metrics + plots)
    mask = _finite_mask(y_true, y_pred)
    dropped = int((~mask).sum())
    y_true_f = y_true[mask]
    y_pred_f = y_pred[mask]

    residuals = y_true_f - y_pred_f
    abs_err = np.abs(residuals)

    # Metrics
    mae = float(mean_absolute_error(y_true_f, y_pred_f))
    rmse = float(np.sqrt(mean_squared_error(y_true_f, y_pred_f)))
    r2 = float(r2_score(y_true_f, y_pred_f))

    metrics: Dict[str, Any] = {
        "problem_type": "regression",
        "n_samples": int(len(y_true)),
        "n_used": int(len(y_true_f)),
        "n_dropped_nonfinite": dropped,
        "mae": mae,
        "rmse": rmse,
        "r2": r2,
        "residual_mean": float(np.mean(residuals)),
        "residual_std": float(np.std(residuals)),
        "abs_error_p50": float(np.percentile(abs_err, 50)),
        "abs_error_p90": float(np.percentile(abs_err, 90)),
        "abs_error_p95": float(np.percentile(abs_err, 95)),
    }

    # --- Plot: Predicted vs Actual ---
    fig, ax = plt.subplots()
    ax.scatter(y_true_f, y_pred_f, alpha=0.5)
    ax.set_xlabel("Actual")
    ax.set_ylabel("Predicted")
    ax.set_title("Predicted vs Actual")

    mn = float(min(np.min(y_true_f), np.min(y_pred_f)))
    mx = float(max(np.max(y_true_f), np.max(y_pred_f)))
    ax.plot([mn, mx], [mn, mx])

    pred_path = reports_dir / f"{cfg.save_prefix}_pred_vs_actual.png"
    fig.tight_layout()
    fig.savefig(pred_path, dpi=150)
    plt.close(fig)
    metrics["pred_vs_actual_path"] = str(pred_path)

    # --- Plot: Residuals vs Predicted ---
    fig, ax = plt.subplots()
    ax.scatter(y_pred_f, residuals, alpha=0.5)
    ax.axhline(0.0)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Residual (Actual - Predicted)")
    ax.set_title("Residuals vs Predicted")

    res_scatter_path = reports_dir / f"{cfg.save_prefix}_residuals_vs_pred.png"
    fig.tight_layout()
    fig.savefig(res_scatter_path, dpi=150)
    plt.close(fig)
    metrics["residuals_vs_pred_path"] = str(res_scatter_path)

    # --- Plot: Residual Histogram ---
    fig, ax = plt.subplots()
    ax.hist(residuals, bins=50)
    ax.set_xlabel("Residual")
    ax.set_ylabel("Count")
    ax.set_title("Residual Distribution")

    res_hist_path = reports_dir / f"{cfg.save_prefix}_residual_hist.png"
    fig.tight_layout()
    fig.savefig(res_hist_path, dpi=150)
    plt.close(fig)
    metrics["residual_hist_path"] = str(res_hist_path)

    # --- Plot: Absolute Error Histogram ---
    fig, ax = plt.subplots()
    ax.hist(abs_err, bins=50)
    ax.set_xlabel("|Residual|")
    ax.set_ylabel("Count")
    ax.set_title("Absolute Error Distribution")

    abs_hist_path = reports_dir / f"{cfg.save_prefix}_abs_error_hist.png"
    fig.tight_layout()
    fig.savefig(abs_hist_path, dpi=150)
    plt.close(fig)
    metrics["abs_error_hist_path"] = str(abs_hist_path)

    # --- Save metrics JSON ---
    metrics_path = reports_dir / f"{cfg.save_prefix}_metrics.json"
    with open(metrics_path, "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2)
    metrics["metrics_path"] = str(metrics_path)

    # --- Save predictions CSV (optional) ---
    if cfg.save_predictions_csv:
        pred_df = pd.DataFrame({"y_true": y_true, "y_pred": y_pred})
        pred_df["residual"] = pred_df["y_true"] - pred_df["y_pred"]
        pred_df["abs_error"] = pred_df["residual"].abs()

        if isinstance(X_test, pd.DataFrame) and cfg.id_column and cfg.id_column in X_test.columns:
            pred_df.insert(0, cfg.id_column, X_test[cfg.id_column].values)

        preds_path = reports_dir / f"{cfg.save_prefix}_predictions.csv"
        pred_df.to_csv(preds_path, index=False)
        metrics["predictions_path"] = str(preds_path)

    return metrics

[evaluate.evaluate_model] Starting evaluation
[evaluate.evaluate_model] Metrics={'pr_auc': 0.2125197710236737, 'roc_auc': 0.5389090909090909}
[notebook] Validation metrics: {'pr_auc': 0.2125197710236737, 'roc_auc': 0.5389090909090909}
[notebook] Test metrics: Skipped to protect the audit vault. Set 'evaluate_on_test': True to run.

💡 Note: evaluate_model automatically returns PR AUC & ROC AUC because problem_type='classification'.


### 10) Inference demo (`src.infer`)

Educational note
- Inference simulates what happens after training
- We deliberately use rows from the test split to simulate unseen cases

In [ ]:
import pandas as pd


def run_inference(model, X_infer: pd.DataFrame) -> pd.DataFrame:
    """
    Inputs:
    - model: fitted sklearn Pipeline.
    - X_infer: DataFrame of features to predict on.
    Outputs:
    - predictions_df: DataFrame with a single column "prediction", preserving
    input index.
    Why this contract matters for reliable ML delivery:
    - Stable output schemas prevent downstream integration failures (APIs,
    batch jobs, dashboards).
    """
    print("[infer.run_inference] Running inference")
    # TODO: replace with logging later

    if X_infer is None or len(X_infer) == 0:
        return pd.DataFrame({"prediction": []},
                            index=getattr(X_infer, "index", None))

    preds = model.predict(X_infer)
    predictions_df = pd.DataFrame({"prediction": preds}, index=X_infer.index)

    # --------------------------------------------------------
    # START STUDENT CODE
    # --------------------------------------------------------
    # TODO_STUDENT: Add post-processing such as thresholding, rounding, or
    # business rules.
    # Why: Many real deployments require translating raw outputs into
    # decision-ready formats.
    # Examples:
    # 1. Clip regression predictions to non-negative values
    # 2. Convert probabilities into risk buckets
    #
    # Optional forcing function (leave commented)
    # raise NotImplementedError("Student: You must implement this logic to
    # proceed!")
    #
    # Placeholder (Remove this after implementing your code):
    print("Warning: Student has not implemented this section yet")
    # --------------------------------------------------------
    # END STUDENT CODE
    # --------------------------------------------------------

    return predictions_df

[infer.run_inference] Running inference
[notebook] Inference results


,prediction,proba
31,1,0.607638
579,1,0.608669
321,0,0.449023
101,0,0.450629
514,0,0.373308
417,0,0.451165
432,0,0.369772
567,1,0.609184
137,1,0.604540
816,0,0.450094


### 11) Inspect production artifacts produced by the orchestrator

This notebook does not write artifacts to disk

Use this cell only after you run the orchestrator from terminal
- `python -m src.main`

Goal
- Prove that the factory output exists on disk
- Inspect outputs without modifying them

In [14]:
from src.utils import load_model

CLEAN_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "clean.csv"
MODEL_PATH = PROJECT_ROOT / "models" / "model.joblib"
PREDICTIONS_PATH = PROJECT_ROOT / "reports" / "predictions.csv"

try:
    clean_from_disk = pd.read_csv(CLEAN_DATA_PATH)
    preds_from_disk = pd.read_csv(PREDICTIONS_PATH)
    model_from_disk = load_model(MODEL_PATH)

    print("clean.csv shape:", clean_from_disk.shape)
    print("predictions.csv shape:", preds_from_disk.shape)
    print("loaded model type:", type(model_from_disk))

    display(preds_from_disk.head(10))

except Exception as e:
    print("Artifacts not found yet or could not be loaded")
    print("Run from terminal: python -m src.main")
    print("Error:", e)

[utils.load_model] Loading model from /home/idiazl/2026_MLOps/2. mlops_modulirized-repo/models/model.joblib
clean.csv shape: (898, 21)
predictions.csv shape: (10, 2)
loaded model type: <class 'sklearn.pipeline.Pipeline'>


,prediction,proba
0,1,0.607638
1,1,0.608669
2,0,0.449023
3,0,0.450629
4,0,0.373308
5,0,0.451165
6,0,0.369772
7,1,0.609184
8,1,0.604540
9,0,0.450094
